In [1]:
import matplotlib.pyplot as plt
import pickle
import numpy as np
import joblib as jl
from tqdm import tqdm

from functions_synthetic_data_generation import *
from functions_synthetic_data_analysis import *
from functions_hmm import *


plt.style.use('../paper.mplstyle')

print(plt.get_backend())
%matplotlib qt5
print(plt.get_backend())

module://matplotlib_inline.backend_inline
qt5agg


# Simulations generations

## Parameters

In [2]:
# Meta-parameters
MAIN_FOLDER_PATH = '/home/david/Documents/code/loop_noiseless_drift_estimation_output'

# Simulation generation parameters

P_CW_REWARD = 0.8
P_CCW_REWARD = 0
P_CW_INIT_RANGE = np.linspace(0.01,0.99,100)
STEPS_NUMBER = 40
NOISE_AMPLITUDE = 0.1
DRIFT_INIT = 0
DRIFT_VALUES_ARR = np.linspace(0.005,0.1,5)

DEFAULT_ARGS_DICT = {'p_cw_reward': P_CW_REWARD, 
             'p_ccw_reward': P_CCW_REWARD, 
             'p_cw_init': 0.5, 
             'steps_number': STEPS_NUMBER, 
             'noise_amplitude': NOISE_AMPLITUDE, 
             'drift_matrix': np.array([[DRIFT_VALUES_ARR[0], -DRIFT_VALUES_ARR[0]],
                                       [0                  , 0                   ]]), 
             'drift_init': DRIFT_INIT}


SIMULATIONS_SET_SIZE = 50
N_SETS = 100
SIMULATIONS_SET_SIZE_LIST = [SIMULATIONS_SET_SIZE]*N_SETS

# Fit parameters
STATES_NUMBER_RANGE = np.arange(2,12)
N_FITS = 50
N_JOBS = 5

## Plots

### Plot HMM

In [3]:
example_drift = 2
simu_index = 0
example_set = 0

with open(f'{MAIN_FOLDER_PATH}/drift_{example_drift}/simulation_set_{example_set}.pkl', 'rb') as file:
    synthetic_data = pickle.load(file)

with open(f'{MAIN_FOLDER_PATH}/drift_{example_drift}/hmm_fit_output_{example_set}.pkl', 'rb') as file:
    fit_output = pickle.load(file)


model = fit_output['models'][1]
choices_sequence = [synth_data['choices'] for synth_data in synthetic_data][simu_index]
rewards_sequence = [synth_data['rewards'] for synth_data in synthetic_data][simu_index]
proba_sequence = [synth_data['p_cw'] for synth_data in synthetic_data][simu_index]

transmat = model.transmat_
startprob = model.startprob_
emissionmat = model.emissionprob_
fig = plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(2, 1, height_ratios=(1,0.1))
row1 = gs[0,0].subgridspec(1, 3, width_ratios=(1,0.5,1))
ax1 = plt.subplot(row1[0,0])
ax2 = plt.subplot(row1[0,1])
ax3 = plt.subplot(row1[0,2])
row2 = gs[1,0].subgridspec(1, 1)
ax = plt.subplot(row2[0,0])

im1 = ax1.imshow(transmat, vmin=0, vmax=1)
ax1.set_xticks(np.arange(len(transmat)))
ax1.set_yticks(np.arange(len(transmat)))
ax1.set_title('Transition matrix', fontsize=5)
ax1.set_xlabel('To state')
ax1.set_ylabel('From state')

im2 = ax2.imshow(np.reshape([startprob],(-1,1)), vmin=0, vmax=1)
ax2.set_xticks([])
ax2.set_yticks(np.arange(len(startprob)))
ax2.set_title('Initial state population', fontsize=5)

im3 = ax3.imshow(emissionmat, vmin=0, vmax=1)
ax3.set_title('Emission matrix', fontsize=5)
ax3.set_xticks((0,1))
ax3.set_xticklabels(('CCW','CW'))

ax3.set_yticks(np.arange(len(emissionmat)))

plt.colorbar(im1,cax=ax, orientation='horizontal')

### Plotting initial probability distribution

In [4]:
example_set = 8
example_drift = 2

with open(f'{MAIN_FOLDER_PATH}/drift_{example_drift}/simulation_set_{example_set}.pkl', 'rb') as file:
    synthetic_data = pickle.load(file)

with open(f'{MAIN_FOLDER_PATH}/drift_{example_drift}/hmm_fit_output_{example_set}.pkl', 'rb') as file:
    fit_output = pickle.load(file)

model = fit_output['models'][6]
test_data = [synth_data['choices'] for synth_data in synthetic_data]
test_proba = [synth_data['p_cw'] for synth_data in synthetic_data]

########################
### Reformating Data ###
########################


emissionmat = model.emissionprob_
startprob = model.startprob_
# # transmat = model.transmat_

# initial_states_arr = np.array(count_initial_state_occurences(model,test_data))

# infered_initial_prob_arr = np.array([])

# for s in initial_states_arr:

#     proba = emissionmat[s,1]
#     infered_initial_prob_arr = np.append(infered_initial_prob_arr, proba)

initial_prob_arr = np.array([])

for p_sequence in test_proba:

    initial_prob_arr = np.append(initial_prob_arr,p_sequence[0])

fig = plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)
ax = plt.subplot(row[0,0])

bins = np.linspace( emissionmat[0,1] - (emissionmat[1,1] - emissionmat[0,1])/2,
                    emissionmat[-1,1] + (emissionmat[1,1] - emissionmat[0,1])/2,
                    len(emissionmat)+1)

binned_initial_prob_arr = np.histogram(initial_prob_arr, bins)[0]/len(initial_prob_arr)

ax.bar(emissionmat[:,1], binned_initial_prob_arr, width=emissionmat[1,1]-emissionmat[0,1], alpha=0.5, label='Simulations initial proba distribution')
ax.bar(emissionmat[:,1], startprob, width=emissionmat[1,1]-emissionmat[0,1], alpha=0.5, label='HMM initial state population')

ax.set_yticks(np.arange(0,1,0.2))
ax.set_ylim((0,0.5))
ax.set_xlabel('Starting probability')
ax.set_ylabel('Fraction')
ax.legend()

### Predict and plot state sequence

In [5]:
example_set = 8
example_drift = 2

with open(f'{MAIN_FOLDER_PATH}/drift_{example_drift}/simulation_set_{example_set}.pkl', 'rb') as file:
    synthetic_data = pickle.load(file)

with open(f'{MAIN_FOLDER_PATH}/drift_{example_drift}/hmm_fit_output_{example_set}.pkl', 'rb') as file:
    fit_output = pickle.load(file)


model = fit_output['models'][0]
simu_index = 2
choices_sequence = [synth_data['choices'] for synth_data in synthetic_data][simu_index]
rewards_sequence = [synth_data['rewards'] for synth_data in synthetic_data][simu_index]
proba_sequence = [synth_data['p_cw'] for synth_data in synthetic_data][simu_index]

emissionmat = model.emissionprob_
startprob = model.startprob_

predicted_states_sequence = model.predict(np.reshape(choices_sequence,(-1,1)))
predicted_proba_sequence = np.array([emissionmat[s,1] for s in predicted_states_sequence])
colors = ('indigo','violet')
colors_sequence = [colors[r] for r in rewards_sequence]

fig = plt.figure(figsize=(7, 2), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)
ax = plt.subplot(row[0,0])

# ax.scatter(np.arange(len(predicted_proba_sequence)), predicted_proba_sequence, marker='+', c='green')
ax.scatter(np.arange(len(proba_sequence)), proba_sequence, marker='+', c=colors_sequence, alpha=0.7)

ax.plot(np.arange(len(predicted_proba_sequence)), predicted_proba_sequence, marker='+', c='green', alpha=0.5)
ax.plot(np.arange(len(proba_sequence)), proba_sequence, color='k', alpha=0.5)

ax.set_yticks(np.arange(0,1.1,0.2))
# ax.set_ylim((0,0.5))
ax.set_xlabel('Starting probability')
ax.set_ylabel('Fraction')
ax.legend()

/tmp/ipykernel_169832/678587660.py:40: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend()


### Compute and plot slope per state

In [6]:
def compute_hmm_slope(transmat, emissionmat):

    slope_list = []
    proba_vect = emissionmat[:,1]

    for i in range(len(transmat)):

        slope = np.sum(transmat[i,:] * proba_vect) - proba_vect[i]
        slope_list.append(slope)

    return slope_list

def compute_hmm_average_next_proba(transmat, emissionmat):

    avrg_proba_list = []
    proba_vect = emissionmat[:,1]

    for i in range(len(transmat)):

        avrg_proba = np.sum(transmat[i,:] * proba_vect)
        avrg_proba_list.append(avrg_proba)

    return avrg_proba_list

example_number_of_states_index = 1
example_drift = 2

fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

stacked_avrg_proba_lists = []

for example_set in np.arange(SIMULATIONS_SET_SIZE)+1:

    with open(f'{MAIN_FOLDER_PATH}/drift_{example_drift}/hmm_fit_output_{example_set}.pkl', 'rb') as file:
        fit_output = pickle.load(file)

    model = fit_output['models'][example_number_of_states_index]

    # slopes_list = compute_hmm_slope(model.transmat_, model.emissionprob_)
    avrg_proba_list = compute_hmm_average_next_proba(model.transmat_, model.emissionprob_)
    stacked_avrg_proba_lists.append(avrg_proba_list)
    # ax1.plot(model.emissionprob_[:,1],slopes_list)
    # ax1.plot(model.emissionprob_[1:-1,1],slopes_list[1:-1], alpha=0.5)
    ax1.plot(model.emissionprob_[:,1],avrg_proba_list[:], alpha=0.1)

avrg_proba = np.mean(stacked_avrg_proba_lists,axis=0)
ax1.plot(model.emissionprob_[:,1],avrg_proba[:], marker='+', alpha=1)
# ax1.plot(model.emissionprob_[1:-1,1],1-avrg_proba[1:-1], marker='+', alpha=1)
ax1.plot(np.linspace(0,1),np.linspace(0,1), color='k', linestyle='--')
ax1.axvline(0.5, linestyle='--', color='k')

ax1.set_xlim((0,1))
ax1.set_ylim((0,1))
ax1.set_xlabel('P_n(CW)')
ax1.set_ylabel('Average P_n+1(CW)')

print((avrg_proba[-2]-avrg_proba[1])/(model.emissionprob_[-2,1]-model.emissionprob_[1,1]))
# print(np.mean(slopes_list))

nan


/tmp/ipykernel_169832/2768778726.py:61: RuntimeWarning: invalid value encountered in scalar divide
  print((avrg_proba[-2]-avrg_proba[1])/(model.emissionprob_[-2,1]-model.emissionprob_[1,1]))


### Compute and plot up, down and stay probability

In [14]:
def compute_hmm_up_proba(transmat):

    up_proba_list = []

    for i in range(len(transmat)):

        up_proba = np.sum(transmat[i,i+1:])
        up_proba_list.append(up_proba)

    return up_proba_list
model.emissionprob_
def compute_hmm_down_proba(transmat):

    down_proba_list = []

    for i in range(len(transmat)):

        down_proba = np.sum(transmat[i,:i])
        down_proba_list.append(down_proba)

    return down_proba_list

def compute_hmm_same_proba(transmat):

    same_proba_list = []

    for i in range(len(transmat)):

        same_proba = transmat[i,i]
        same_proba_list.append(same_proba)

    return same_proba_list

example_number_of_states_index = 7
example_drift = 2

fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])
# ax2 = plt.subplot(row[1,0])

stacked_up_proba_list = []
stacked_down_proba_list = []
stacked_same_proba_list = []
stacked_transmats = []

for example_set in np.arange(SIMULATIONS_SET_SIZE)+1:

    with open(f'{MAIN_FOLDER_PATH}/drift_{example_drift}/hmm_fit_output_{example_set}.pkl', 'rb') as file:
        fit_output = pickle.load(file)

    model = fit_output['models'][example_number_of_states_index]

    up_proba_list = compute_hmm_up_proba(model.transmat_)
    down_proba_list = compute_hmm_down_proba(model.transmat_)
    same_proba_list = compute_hmm_same_proba(model.transmat_)

    stacked_up_proba_list.append(up_proba_list)
    stacked_down_proba_list.append(down_proba_list)
    stacked_same_proba_list.append(same_proba_list)
    stacked_transmats.append(model.transmat_)


ax1.plot(model.emissionprob_[:,1], np.mean(stacked_up_proba_list, axis=0), marker='+', label="P_n+1(CW) > P_n(CW)")
ax1.plot(model.emissionprob_[:,1], np.mean(stacked_down_proba_list, axis=0), marker='+', label="P_n+1(CW) < P_n(CW)")
ax1.plot(model.emissionprob_[:,1], np.mean(stacked_same_proba_list, axis=0), marker='+', label="P_n+1(CW) = P_n(CW)")
ax1.axhline(0.5, color='grey',linestyle='--')
ax1.set_xlim([0,1])
ax1.set_ylim([0,1])
ax1.set_title(f"{len(model.emissionprob_)} states")
ax1.legend()
#########

# average_transmat = np.mean(stacked_transmats, axis=0)
# average_next_proba = compute_hmm_average_next_proba(model.transmat_, model.emissionprob_)

# ax2.plot(model.emissionprob_[:,1], average_next_proba, marker='+')
# ax2.axhline(0., color='grey',linestyle='--')
# ax2.set_xlim([0,1])
# ax2.set_xlabel("Probability (given by the state)")
# ax2.set_ylabel("Average P_n+1(CW)\ngiven P_n(CW)")

# ax2.set_ylim([0,1])

print(np.mean(stacked_up_proba_list,axis=0))

[0.76573388 0.63459813 0.61034339 0.58551519 0.53364062 0.49844865
 0.31837184 0.30185609 0.        ]


In [8]:

example_set = 1
mean_slopes_list = []
for i in range(SIMULATIONS_SET_SIZE):

    with open(f'{MAIN_FOLDER_PATH}/drift_{2}/hmm_fit_output_{i}.pkl', 'rb') as file:
        temp_fit_output = pickle.load(file)

    model = temp_fit_output['models'][9]
    test_data = [synth_data['choices'] for synth_data in synthetic_data]
    test_proba = [synth_data['p_cw'] for synth_data in synthetic_data]

    slopes_list = compute_hmm_slope(model.transmat_, model.emissionprob_)

    mean_slopes_list.append(np.mean(slopes_list))

fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

ax1.hist(mean_slopes_list,bins=20)

(array([1., 1., 1., 0., 0., 2., 2., 2., 2., 5., 4., 4., 6., 6., 0., 4., 3.,
        3., 0., 4.]),
 array([-0.12721507, -0.11198102, -0.09674698, -0.08151293, -0.06627888,
        -0.05104484, -0.03581079, -0.02057674, -0.0053427 ,  0.00989135,
         0.0251254 ,  0.04035944,  0.05559349,  0.07082754,  0.08606158,
         0.10129563,  0.11652968,  0.13176372,  0.14699777,  0.16223182,
         0.17746586]),
 <BarContainer object of 20 artists>)

### Plot score vs. number of states

In [9]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(2, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

best_states_nbr_list = []

for i in range(SIMULATIONS_SET_SIZE):

    save_path = f'{MAIN_FOLDER_PATH}/drift_{2}/hmm_fit_output_{i}.pkl'

    with open(save_path, 'rb') as file:
            fit_output = pickle.load(file)

    ax1.plot(STATES_NUMBER_RANGE, fit_output['scores'], alpha=0.2)
    # ax1.scatter(len(fit_output[i][0].transmat_), fit_output[i][1], marker='+', alpha=0.2)

    best_states_nbr_list.append(STATES_NUMBER_RANGE[np.argmax(fit_output['scores'])])

ax1.set_xticks(STATES_NUMBER_RANGE)
ax1.set_xlabel("Number of states")
ax1.set_ylabel("log fit score")

ax2 = plt.subplot(row[1,0])

ax2.hist(best_states_nbr_list, bins=STATES_NUMBER_RANGE, align='left')
ax2.set_xlim([1,STATES_NUMBER_RANGE[-1]+1])
ax2.set_xlabel('Number of states')
ax2.set_ylabel('Number of HMM')


Text(0, 0.5, 'Number of HMM')

### Plot average matrix

In [10]:
index_nb_of_states = 3

fixed_state_models_list = []
stacked_init_state_prob = []

for j in range(SIMULATIONS_SET_SIZE):

    with open(f'{MAIN_FOLDER_PATH}/drift_{4}/hmm_fit_output_{j}.pkl', 'rb') as file:
        fit_output = pickle.load(file)

    transmat_to_add = fit_output['models'][index_nb_of_states].transmat_
    init_state_prob_to_add = fit_output['models'][index_nb_of_states].startprob_
    fixed_state_models_list.append(transmat_to_add)
    stacked_init_state_prob.append(init_state_prob_to_add)

average_transmat = np.mean(fixed_state_models_list, axis=0)
var_transmat = np.std(fixed_state_models_list, axis=0)
average_init_state_prob = np.mean(stacked_init_state_prob, axis=0)

fig=plt.figure(figsize=(3.5, 3), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[0].subgridspec(1,4, width_ratios=(1,1,0.1,0.1))
ax1 = plt.subplot(row[0,0])
ax2 = plt.subplot(row[0,1])
ax3 = plt.subplot(row[0,2])
ax4 = plt.subplot(row[0,3])

# ax1.imshow(average_transmat, vmin=0, vmax=1)

im1 = ax1.imshow(average_transmat, vmin=0, vmax=np.max(average_transmat))
ax1.set_xticks(np.arange(len(average_transmat)))
ax1.set_yticks(np.arange(len(average_transmat)))
ax1.set_title('Average transition matrix\nover 100 simulations sets', fontsize=5)
ax1.set_xlabel('To state')
ax1.set_ylabel('From state')

im2 = ax2.imshow(var_transmat, vmin=0, vmax=np.max(average_transmat))
ax2.set_xticks(np.arange(len(average_transmat)))
ax2.set_yticks(np.arange(len(average_transmat)))
ax2.set_title('Standard deviation of transition matrices\nover 100 simulations sets', fontsize=5)

im3 = ax3.imshow(np.reshape(average_init_state_prob,(-1,1)), vmin=0, vmax=np.max(average_transmat))
ax3.set_xticks([])
ax3.set_yticks(np.arange(len(average_transmat)))
# ax3.set_title('Standard deviation of transition matrices\nover 100 simulations sets', fontsize=5)


plt.colorbar(im2,cax=ax4)

In [11]:
# for i in range(len(n_simulations_list)):

#     with open(f'{MAIN_FOLDER_PATH}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index+i+1}.pkl', 'rb') as file:
#         temp_fit_output = pickle.load(file)

#     model = temp_fit_output['models'][2]

#     print(model.monitor_.converged)